# Predictive Model Dataset Building
Now that the KModes clustering work is complete for diagnoses, it is time to assemble and engineer the final dataset to ready it for predictive modeling. Due to the data modeling work at the project's outset, the dataset is mostly in the format I want. I'll use the SQL call to do some custom boolean rendering on categorical fields, do more advanced work through pandas dataframe manipualtion for categoricals with lots of options, and segment the dataset for modeling.

In [1]:
%%bigquery claims_dataset
select
      fc.clm_id
    , fc.is_inpatient_claim
    , fc.is_outpatient_claim
    , fc.is_durable_medical_equipment_claim
    , fc.is_hospice_claim
    , fc.is_home_health_agency_claim
    , fc.is_skilled_nursing_facility_claim
    , fc.claim_start_year
    , fc.claim_start_month
    , fc.claim_end_year
    , fc.carr_clm_prvdr_asgnmt_bool
    , fc.claim_is_in_hospital
    , case
        when fc.clm_freq_cat = 'Admit Through Discharge'
          then 1
        else 0
        end                                               as claim_is_admit_thru_discharge
    , case
        when fc.clm_freq_cat = 'Final Claim'
          then 1
        else 0
        end                                               as claim_is_final
    , fc.claim_has_external_cause
    , extract(year from fc.clm_admsn_dt)                  as claim_admission_year
    , extract(year from fc.nch_bene_dschrg_dt)            as claim_discharge_year
    , extract(year from fc.clm_hospc_start_dt_id)         as claim_hospice_start_year
    , fc.admission_is_emergency
    , fc.admission_is_urgent
    , fc.admission_is_elective
    , case
        when fc.clm_src_ip_admsn_cat = 'Physician Referral'
          then 1
        else 0
        end                                               as inpatient_ref_from_phys
    , case
        when fc.clm_src_ip_admsn_cat = 'Clinic Referral'
          then 1
        else 0
        end                                               as inpatient_ref_from_clinic
    , case
        when fc.clm_src_ip_admsn_cat = 'Transfer from Hospital'
          then 1
        when fc.clm_src_ip_admsn_cat = 'Transfer from SNF'
          then 1
        else 0
        end                                               as inpatient_ref_from_transfer
    , cwc.cluster_names_anemia                            as clu_anemia
    , cwc.cluster_names_cardiac                           as clu_cardiac
    , cwc.cluster_names_chronic_kidney                    as clu_chronic_kidney
    , cwc.cluster_names_depression                        as clu_depression
    , cwc.cluster_names_diab_complications                as clu_diab_complication
    , cwc.cluster_names_diabetes                          as clu_diabetes
    , cwc.cluster_names_gen_complications                 as cluster_names_gen_complication
    , cwc.cluster_names_encephalopathy                    as clu_encephalopathy
    , cwc.cluster_names_insulin_resistance                as clu_insulin_resistance
    , cwc.cluster_names_neonatal_hypertension             as clu_neonatal_hypertension
    , cwc.cluster_names_severe_kidney                     as clu_severe_kidney
    , cwc.cluster_names_stress                            as clu_stress
    , cwc.cluster_names_socioeconomic_obesity_other       as clu_socio_obese_oth
    , cwc.cluster_names_surg_complications                as clu_surg_complication
    , fc.organization_zip                                 as prov_zip
    , coalesce(fc.specialty_group,'No Specialty Group')   as prov_specialty_group
    , coalesce(fc.specialization, 'No Specialization')    as prov_specialization
    , fc.zip_cd                                           as beneficiary_zip
    , fc.beneficiary_is_deceased
    , fc.beneficiary_age_at_claim
    , fc.beneficiary_is_female
    , fc.beneficiary_is_not_white
    , extract(year from fc.covstart)                      as beneficiary_year_of_coverage_start
    , fc.qualified_oasi_curr                              as curr_qualified_old_age_surv
    , fc.qualified_dib_curr                               as curr_qualified_disability
    , fc.qualified_esrd_curr                              as curr_qualified_due_to_renal_disease
    , fc.enrolled_part_c
    , fc.is_primary_claimant
    , fc.has_rds_cvrg
    , fc.has_employer_subsidy_for_month
    , fc.has_full_dual_status
    , fc.has_partial_dual_status
    , fc.cost_share_premium_subsidy_percent
    , case
        when fc.cost_share_copay_cat = 'high copay'
          then 1
        else 0
        end                                               as has_high_copay
    , case
        when fc.cost_share_copay_cat = 'high copay'
          then 1
        when fc.cost_share_copay_cat = 'low copay'
          then 1
        else 0
        end                                               as has_copay
    , fc.claim_duration_in_days
    , fc.clm_pmt_amt
    , fc.clm_pmt_per_rank_tot
    , fc.clm_pmt_per_rank_file
    , fc.clm_tot_chrg_amt
    , fc.clm_hha_tot_visit_cnt
    , fc.nch_bene_ip_ddctbl_amt
    , fc.length_of_coverage_at_claim_in_years
from `spatial-earth-449020-m3.capstone_data_model.fact_claims` fc
left join `spatial-earth-449020-m3.lexie_custom_datasets.claims_with_clusters` cwc
  on fc.clm_id = cwc.clm_id
where
  fc.clm_drg_outlier_stay_cat = 'No Outlier' --remove cost outliers, only 183 in entire data set

Query is running:   0%|          |

Downloading:   0%|          |

In [2]:
claims_dataset.head()

,clm_id,is_inpatient_claim,is_outpatient_claim,is_durable_medical_equipment_claim,is_hospice_claim,is_home_health_agency_claim,is_skilled_nursing_facility_claim,claim_start_year,claim_start_month,claim_end_year,...,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_pmt_per_rank_tot,clm_pmt_per_rank_file,clm_tot_chrg_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years
0,-10000931486115,False,False,True,False,False,False,2020,1,2020,...,0,0,0,6.56,0.9456,0.3019,0.0,0,0.0,53
1,-10000931486121,False,False,True,False,False,False,2020,2,2020,...,0,0,0,6.56,0.9456,0.3019,0.0,0,0.0,53
2,-10000931486137,False,False,True,False,False,False,2022,7,2022,...,0,0,0,6.56,0.9456,0.3019,0.0,0,0.0,55
3,-10000931486113,False,False,True,False,False,False,2019,6,2019,...,0,0,0,6.56,0.9456,0.3019,0.0,0,0.0,52
4,-10000931486110,False,False,True,False,False,False,2018,6,2018,...,0,0,0,6.56,0.9456,0.3019,0.0,0,0.0,51


In [3]:
spec_grp_rework = claims_dataset[['clm_id','prov_specialty_group']]
group_cnt = spec_grp_rework.groupby('prov_specialty_group')['prov_specialty_group'].count().reset_index(name='counts').sort_values('counts',ascending=False)
group_cnt

,prov_specialty_group,counts
3,Hospitals,246673
2,Hospital Units,113554
1,Ambulatory Health Care Facilities,95170
6,No Specialty Group,77479
0,Agencies,12261
7,Nursing & Custodial Care Facilities,9776
5,Managed Care Organizations,73
8,Residential Treatment Facilities,26
10,Transportation Services,12
9,Suppliers,8


I want to create custom boolean fields, eliminating booleans for No Specialty Group, Managed Care Organizations, Residential Treatment Facilities, Transportation Services, Suppliers, and Laboratories due to low numbers.

In [4]:
import pandas as pd

#first render categorical text as better column name format
spec_grp_rework['prov_specialty_group'] = spec_grp_rework['prov_specialty_group'].str.replace(' ','_')
spec_grp_rework['prov_specialty_group'] = spec_grp_rework['prov_specialty_group'].str.lower()

#then create dummies
spec_grp_rework = pd.get_dummies(spec_grp_rework,prefix_sep='_',prefix='prov_spec_group_is')
spec_grp_rework.head()

<ipython-input-4-a22c1fb7d829>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spec_grp_rework['prov_specialty_group'] = spec_grp_rework['prov_specialty_group'].str.replace(' ','_')
<ipython-input-4-a22c1fb7d829>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spec_grp_rework['prov_specialty_group'] = spec_grp_rework['prov_specialty_group'].str.lower()


,clm_id,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_laboratories,prov_spec_group_is_managed_care_organizations,prov_spec_group_is_no_specialty_group,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_group_is_residential_treatment_facilities,prov_spec_group_is_suppliers,prov_spec_group_is_transportation_services
0,-10000931486115,False,False,False,False,False,False,True,False,False,False,False
1,-10000931486121,False,False,False,False,False,False,True,False,False,False,False
2,-10000931486137,False,False,False,False,False,False,True,False,False,False,False
3,-10000931486113,False,False,False,False,False,False,True,False,False,False,False
4,-10000931486110,False,False,False,False,False,False,True,False,False,False,False


In [5]:
#finally, drop columns no longer needed
spec_group_to_drop = ['prov_spec_group_is_no_specialty_group','prov_spec_group_is_managed_care_organizations',
                      'prov_spec_group_is_residential_treatment_facilities','prov_spec_group_is_transportation_services',
                      'prov_spec_group_is_suppliers','prov_spec_group_is_laboratories']
spec_grp_final = spec_grp_rework.drop(columns=spec_group_to_drop)
spec_grp_final.head()

,clm_id,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities
0,-10000931486115,False,False,False,False,False
1,-10000931486121,False,False,False,False,False
2,-10000931486137,False,False,False,False,False
3,-10000931486113,False,False,False,False,False
4,-10000931486110,False,False,False,False,False


In [6]:
#merge the new specialty group column into the master dataset and drop original column

claims_with_spec_grp = pd.merge(claims_dataset, spec_grp_final, how='left',on='clm_id')
claims_with_spec_grp = claims_with_spec_grp.drop('prov_specialty_group',axis=1)
claims_with_spec_grp.columns

Index(['clm_id', 'is_inpatient_claim', 'is_outpatient_claim',
       'is_durable_medical_equipment_claim', 'is_hospice_claim',
       'is_home_health_agency_claim', 'is_skilled_nursing_facility_claim',
       'claim_start_year', 'claim_start_month', 'claim_end_year',
       'carr_clm_prvdr_asgnmt_bool', 'claim_is_in_hospital',
       'claim_is_admit_thru_discharge', 'claim_is_final',
       'claim_has_external_cause', 'claim_admission_year',
       'claim_discharge_year', 'claim_hospice_start_year',
       'admission_is_emergency', 'admission_is_urgent',
       'admission_is_elective', 'inpatient_ref_from_phys',
       'inpatient_ref_from_clinic', 'inpatient_ref_from_transfer',
       'clu_anemia', 'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',

I'll finish my feature engineering by performing a similar custom boolean exercise on specialization.

In [7]:
spec_rework = claims_dataset[['clm_id','prov_specialization']]
spec_cnt = spec_rework.groupby('prov_specialization')['prov_specialization'].count().reset_index(name='counts').sort_values('counts',ascending=False)
spec_cnt

,prov_specialization,counts
28,No Specialization,445200
16,Federally Qualified Health Center (FQHC),30450
5,Ambulatory Surgical,15785
8,Critical Access,14833
47,Urgent Care,11671
45,Rural Health,5678
12,Emergency Care,5343
3,Adult Mental Health,4087
44,Rural,2723
27,Multi-Specialty,2473


There are a lot more categories in specialization than specialty group! I'm going to cut off types that have fewer than 500 values. This is not likely to be a hugely important indicator considering most providers do not have a specialization recorded.

In [8]:
spec_to_drop = ['prov_spec_is_no_specialization','prov_spec_is_militaryus_coast_guard_outpatient',
                'prov_spec_is_medical_specialty',
                'prov_spec_is_adult_day_care','prov_spec_is_methadone','prov_spec_is_radiology',
                'prov_spec_is_endoscopy','prov_spec_is_lithotripsy','prov_spec_is_family_planning_nonsurgical',
                'prov_spec_is_developmental_disabilities','prov_spec_is_hearing_and_speech',
                'prov_spec_is_ambulatory_family_planning_facility',
                'prov_spec_is_public_health_federal','prov_spec_is_pain','prov_spec_is_sleep_disorder_diagnostic',
                'prov_spec_is_occupational_medicine','prov_spec_is_nursing_care_pediatric','prov_spec_is_public_health_state_or_local',
                'prov_spec_is_infusion_therapy','prov_spec_is_women','prov_spec_is_migrant_health','prov_spec_is_adolescent_and_children_mental_health',
                'prov_spec_is_radiology_mobile_mammography','prov_spec_is_oral_and_maxillofacial_surgery','prov_spec_is_adult_care_home',
                'prov_spec_is_magnetic_resonance_imaging_mri']

In [9]:
#first render categorical text as better column name format - stripping special characters in addition to replacing spaces
spec_rework['prov_specialization'] = spec_rework['prov_specialization'].replace(to_replace=r'[&|/|,|-|(|)|.|-]',value='',regex=True)

#next, group similar categories for minimal dummy columns
spec_rework = spec_rework.replace(to_replace='Rural Health',value='Rural')
spec_rework = spec_rework.replace(to_replace='Critical Access Hospital',value='Critical Access')
spec_rework = spec_rework.replace(to_replace='Rehabilitation Comprehensive Outpatient Rehabilitation Facility CORF',value='Rehabilitation')
spec_rework = spec_rework.replace(to_replace='Rehabilitation Cardiac Facilities',value='Rehabilitation')
spec_rework = spec_rework.replace(to_replace='Rehabilitation Substance Use Disorder',value='Rehabilitation')
spec_rework = spec_rework.replace(to_replace='EndStage Renal Disease ESRD Treatment',value='ESRD')

spec_cnt = spec_rework.groupby('prov_specialization')['prov_specialization'].count().reset_index(name='counts').sort_values('prov_specialization')
spec_cnt


<ipython-input-9-6f41fdc0e52d>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spec_rework['prov_specialization'] = spec_rework['prov_specialization'].replace(to_replace=r'[&|/|,|-|(|)|.|-]',value='',regex=True)


,prov_specialization,counts
0,Adolescent and Children Mental Health,7
1,Adult Care Home,5
2,Adult Day Care,215
3,Adult Mental Health,4087
4,Ambulatory Family Planning Facility,49
5,Ambulatory Surgical,15785
6,Children,2108
7,Community Health,1798
8,Critical Access,16129
9,Dental,734


In [10]:
#render columns into more friendly names
spec_rework['prov_specialization'] = spec_rework['prov_specialization'].str.replace(' ','_')
spec_rework['prov_specialization'] = spec_rework['prov_specialization'].str.lower()

#then create dummies
spec_rework = pd.get_dummies(spec_rework,prefix_sep='_',prefix='prov_spec_is')

In [11]:
#drop unneeded columns
spec_final = spec_rework.drop(columns=spec_to_drop)
spec_final.columns

Index(['clm_id', 'prov_spec_is_adult_mental_health',
       'prov_spec_is_ambulatory_surgical', 'prov_spec_is_children',
       'prov_spec_is_community_health', 'prov_spec_is_critical_access',
       'prov_spec_is_dental', 'prov_spec_is_emergency_care',
       'prov_spec_is_esrd',
       'prov_spec_is_federally_qualified_health_center_fqhc',
       'prov_spec_is_health_service',
       'prov_spec_is_mental_health_including_community_mental_health_center',
       'prov_spec_is_multispecialty', 'prov_spec_is_oncology',
       'prov_spec_is_physical_therapy', 'prov_spec_is_primary_care',
       'prov_spec_is_rehabilitation', 'prov_spec_is_rural',
       'prov_spec_is_urgent_care'],
      dtype='object')

In [12]:
#merge the new specialty columns into the master dataset and drop original column

claims_with_spec = pd.merge(claims_with_spec_grp, spec_final, how='left',on='clm_id')
claims_with_spec = claims_with_spec.drop('prov_specialization',axis=1)
claims_with_spec.columns

Index(['clm_id', 'is_inpatient_claim', 'is_outpatient_claim',
       'is_durable_medical_equipment_claim', 'is_hospice_claim',
       'is_home_health_agency_claim', 'is_skilled_nursing_facility_claim',
       'claim_start_year', 'claim_start_month', 'claim_end_year',
       'carr_clm_prvdr_asgnmt_bool', 'claim_is_in_hospital',
       'claim_is_admit_thru_discharge', 'claim_is_final',
       'claim_has_external_cause', 'claim_admission_year',
       'claim_discharge_year', 'claim_hospice_start_year',
       'admission_is_emergency', 'admission_is_urgent',
       'admission_is_elective', 'inpatient_ref_from_phys',
       'inpatient_ref_from_clinic', 'inpatient_ref_from_transfer',
       'clu_anemia', 'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',

In [13]:
#peeking at the dataframe to see overall statistics
pd.set_option('display.max_columns',None)
claims_with_spec.describe(include='all')

,clm_id,is_inpatient_claim,is_outpatient_claim,is_durable_medical_equipment_claim,is_hospice_claim,is_home_health_agency_claim,is_skilled_nursing_facility_claim,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_pmt_per_rank_tot,clm_pmt_per_rank_file,clm_tot_chrg_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,555034.0,555034,555034,555034,555034,555034,555034,555034.0,555034.0,555034.0,555034,555034,555034.0,555034.0,555034,555034.0,555034.0,555034.0,555034,555034,555034,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,477555,555034.0,555034,555034.000000,555034,555034,555034.0,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034.0,555034.0,555034.0,555034.0,555034.000000,555034.000000,555034.000000,555034.000000,555034.0,555034.000000,555034.0,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034
unique,<NA>,2,2,2,2,2,2,<NA>,<NA>,<NA>,2,2,<NA>,<NA>,2,<NA>,<NA>,<NA>,2,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,7293,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
top,<NA>,False,True,False,False,False,False,<NA>,<NA>,<NA>,False,True,<NA>,<NA>,True,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,352427509,<NA>,False,NaN,False,False,<NA>,False,False,False,True,False,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,534350,402653,517252,553949,554541,553402,<NA>,<NA>,<NA>,426547,423337,<NA>,<NA>,415105,<NA>,<NA>,<NA>,538669,554374,549743,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8313,<NA>,555034,NaN,283405,339093,<NA>,282222,387988,392271,347555,301542,553852,553926,453463,517986,<NA>,<NA>,<NA

I'm going to drop a column that will be too closely correlated with the predictor and fill NAs to 0 before writing my working dataset to BigQuery. Since I am predicting clm_pmt_amt (the amount Medicare will pay for the claim), having clm_tot_chrg_amt will be too close to the predicted value to provide a good predictive model. I also want to drop the percentile ranks of claim payments for the same reason.

In [14]:
claim_pred_all_final = claims_with_spec.drop('clm_tot_chrg_amt',axis=1)
claim_pred_all_final = claim_pred_all_final.drop('clm_pmt_per_rank_tot',axis=1)
claim_pred_all_final = claim_pred_all_final.drop('clm_pmt_per_rank_file',axis=1)
claim_pred_all_final = claim_pred_all_final.fillna(0)
claim_pred_all_final['prov_zip'] = (claim_pred_all_final['prov_zip'].astype(int))

claim_pred_all_final.describe(include='all')

,clm_id,is_inpatient_claim,is_outpatient_claim,is_durable_medical_equipment_claim,is_hospice_claim,is_home_health_agency_claim,is_skilled_nursing_facility_claim,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,555034.0,555034,555034,555034,555034,555034,555034,555034.0,555034.0,555034.0,555034,555034,555034.0,555034.0,555034,555034.0,555034.0,555034.0,555034,555034,555034,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,555034.0,5.550340e+05,555034.0,555034,555034.000000,555034,555034,555034.0,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034.0,555034.0,555034.0,555034.0,555034.000000,555034.0,555034.000000,555034.0,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034,555034
unique,<NA>,2,2,2,2,2,2,<NA>,<NA>,<NA>,2,2,<NA>,<NA>,2,<NA>,<NA>,<NA>,2,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
top,<NA>,False,True,False,False,False,False,<NA>,<NA>,<NA>,False,True,<NA>,<NA>,True,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,False,False,<NA>,False,False,False,True,False,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,534350,402653,517252,553949,554541,553402,<NA>,<NA>,<NA>,426547,423337,<NA>,<NA>,415105,<NA>,<NA>,<NA>,538669,554374,549743,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,555034,NaN,283405,339093,<NA>,282222,387988,392271,347555,301542,553852,553926,453463,517986,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,542773,459864,441480,308361,545258,550947,539249,552926,553236,538905,554300,549691,553357,524584,55419

In [ ]:
from pandas_gbq import to_gbq

project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.working_pred_set'

# Write DataFrame to BigQuery
to_gbq(claim_pred_all_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 8439.24it/s]


Finally, I'll segment the master set into the "claim type" booleans. Since claims range from medical devices and outpatient visits to hospital and hospice stays, clm_pmt_amt ranges widely. To have the most predictive power, I'll predict amount by the type of claim presented. These booleans derive from the seven claims files aggregated into the fact_claims table in our data model.

Once I segment, I'll remove the filtering booleans from each subset and perform a final describe check on columns (as a note, not all variables are used for all claims, so variables with only one possible value or very skewed values per claim type will be eliminated). The goal will be to have largely the same variables for each model, with a handful of special fields for specific claim types.

In [15]:
claim_type_filters = ['is_inpatient_claim','is_outpatient_claim','is_durable_medical_equipment_claim'
,'is_hospice_claim','is_home_health_agency_claim','is_skilled_nursing_facility_claim']

In [39]:
claim_inpatient = claim_pred_all_final[claim_pred_all_final['is_inpatient_claim']==True]
claim_inpatient = claim_inpatient.drop(claim_type_filters,axis=1)
claim_inpatient.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,20684.0,20684.0,20684.0,20684.0,20684,20684,20684.0,20684.0,20684,20684.0,20684.0,20684.0,20684,20684,20684,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,20684.0,2.068400e+04,20684.0,20684,20684.000000,20684,20684,20684.0,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684.0,20684.0,20684.0,20684.0,20684.000000,20684.0,20684.000000,20684.0,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684,20684
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,2,<NA>,<NA>,<NA>,2,2,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
top,<NA>,<NA>,<NA>,<NA>,False,True,<NA>,<NA>,True,<NA>,<NA>,<NA>,True,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,False,False,<NA>,True,False,False,True,True,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,20684,20684,<NA>,<NA>,17362,<NA>,<NA>,<NA>,16365,20024,17025,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,20684,NaN,11314,13125,<NA>,13066,13855,19567,12834,16069,20615,20617,16541,19187,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,15739,17509,19865,17299,13579,20661,20434,20676,20620,20451,20627,20538,20660,18996,20485,20678,20609,20675,20677,20550,20649,20431,20609
mean,-10000930748765.527344,2019.087749,6.522723,2019.090892,NaN,NaN,1.0,0.0,NaN,2019.087749,2019.090892,1.0,NaN,NaN,NaN,0.249565,0.250242,0.500193,0.539016,0.545687,0.325662,0.016583,0.558789,0.243038,0.22046,0.009476,0.500193,0.332189,0.09853,0.556662,1.0,0.839393,4.543970e+08,43444.762377,NaN,66.358427,NaN,NaN,2000.432702,NaN,NaN,NaN,NaN,NaN,N

In [40]:
claim_ip_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital','claim_is_admit_thru_discharge',
                 'claim_is_final','claim_hospice_start_year','has_rds_cvrg','has_employer_subsidy_for_month',
                 'clm_hha_tot_visit_cnt','prov_spec_is_adult_mental_health','prov_spec_is_children','prov_spec_is_community_health','prov_spec_is_dental',
                 'prov_spec_is_esrd','prov_spec_is_mental_health_including_community_mental_health_center','prov_spec_is_multispecialty','prov_spec_is_oncology',
                 'prov_spec_is_physical_therapy','prov_spec_is_rehabilitation','prov_spec_is_urgent_care','beneficiary_is_deceased']

In [41]:
claim_ip_final = claim_inpatient.drop(claim_ip_drop,axis=1)
claim_ip_final.columns

Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'claim_has_external_cause', 'claim_admission_year',
       'claim_discharge_year', 'admission_is_emergency', 'admission_is_urgent',
       'admission_is_elective', 'inpatient_ref_from_phys',
       'inpatient_ref_from_clinic', 'inpatient_ref_from_transfer',
       'clu_anemia', 'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'prov_zip', 'beneficiary_zip',
       'beneficiary_age_at_claim', 'beneficiary_is_female',
       'beneficiary_is_not_white', 'beneficiary_year_of_coverage_start',
       'curr_qualified_old_age_surv', 'curr_qualified_disability',
       'curr_qualified_due_to_renal_disease', 'enrolled_part_c',
       'is_primar

In [42]:
claim_outpatient = claim_pred_all_final[claim_pred_all_final['is_outpatient_claim']==True]
claim_outpatient = claim_outpatient.drop(claim_type_filters,axis=1)
claim_outpatient.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,402653.0,402653.0,402653.0,402653.0,402653,402653,402653.0,402653.0,402653,402653.0,402653.0,402653.0,402653,402653,402653,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,402653.0,4.026530e+05,402653.0,402653,402653.000000,402653,402653,402653.0,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653.0,402653.0,402653.0,402653.0,402653.000000,402653.0,402653.0,402653.0,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653,402653
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,2,<NA>,<NA>,<NA>,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,2,2,2,2,2,2,2,2
top,<NA>,<NA>,<NA>,<NA>,False,True,<NA>,<NA>,True,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,True,False,<NA>,False,False,False,True,False,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,402653,402653,<NA>,<NA>,396367,<NA>,<NA>,<NA>,402653,402653,402653,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,402653,NaN,202760,243586,<NA>,223496,290403,246274,254780,273620,401956,402019,329552,375674,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,398601,362228,297031,230920,401813,398691,387894,400634,402595,387777,402356,397895,401352,402653,402653,401886,400582,401899,401905,402638,400047,399409,401004
mean,-10000930758672.804688,2019.228025,6.460496,2019.228241,NaN,NaN,1.0,0.0,NaN,1.0,1.0,1.0,NaN,NaN,NaN,0.0,0.0,0.0,0.583969,0.421475,0.699443,0.05729,0.788217,0.400116,0.142075,0.045404,0.518444,0.455708,0.637134,0.58607,0.999868,0

In [43]:
claim_op_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital','claim_is_admit_thru_discharge',
                 'claim_is_admit_thru_discharge','claim_is_final',
                 'claim_admission_year','claim_discharge_year','claim_hospice_start_year',
                 'admission_is_emergency','admission_is_urgent','admission_is_elective','inpatient_ref_from_phys','inpatient_ref_from_clinic','inpatient_ref_from_transfer',
                 'beneficiary_is_deceased','clm_hha_tot_visit_cnt',
                 'nch_bene_ip_ddctbl_amt',
                 'prov_spec_is_community_health','prov_spec_is_dental','prov_spec_is_federally_qualified_health_center_fqhc','prov_spec_is_health_service',
                 'prov_spec_is_primary_care']

claim_op_final = claim_outpatient.drop(claim_op_drop,axis=1)
claim_op_final.columns


Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'claim_has_external_cause', 'clu_anemia', 'clu_cardiac',
       'clu_chronic_kidney', 'clu_depression', 'clu_diab_complication',
       'clu_diabetes', 'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'prov_zip', 'beneficiary_zip',
       'beneficiary_age_at_claim', 'beneficiary_is_female',
       'beneficiary_is_not_white', 'beneficiary_year_of_coverage_start',
       'curr_qualified_old_age_surv', 'curr_qualified_disability',
       'curr_qualified_due_to_renal_disease', 'enrolled_part_c',
       'is_primary_claimant', 'has_rds_cvrg', 'has_employer_subsidy_for_month',
       'has_full_dual_status', 'has_partial_dual_status',
       'cost_share_premium_subsidy_percent', 'has_high_copay', 'has_copay',
       'claim_duration_in_days', 'clm_pm

In [44]:
claim_dme = claim_pred_all_final[claim_pred_all_final['is_durable_medical_equipment_claim']==True]
claim_dme = claim_dme.drop(claim_type_filters,axis=1)
claim_dme.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,37782.0,37782.0,37782.0,37782.0,37782,37782,37782.0,37782.0,37782,37782.0,37782.0,37782.0,37782,37782,37782,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782.0,37782,37782.00000,37782,37782,37782.0,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782.0,37782.0,37782.0,37782.0,37782.000000,37782.0,37782.0,37782.0,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,1,<NA>,<NA>,<NA>,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
top,<NA>,<NA>,<NA>,<NA>,True,False,<NA>,<NA>,False,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,False,False,<NA>,True,False,False,True,True,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,37782,37782,<NA>,<NA>,37782,<NA>,<NA>,<NA>,37782,37782,37782,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,37782,NaN,21781,23985,<NA>,21067,21117,36886,22636,32793,37699,37701,30706,35280,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782,37782
mean,-10000930748506.376953,2018.975359,6.493886,2018.975359,NaN,NaN,0.0,0.0,NaN,1.0,1.0,1.0,NaN,NaN,NaN,0.0,0.0,0.0,0.416653,0.392118,0.173495,0.011249,0.244217,0.115425,0.083876,0.023874,0.352655,0.210391,0.043513,0.580038,1.0,0.00405,0.0,42480.245567,NaN,64.93129,NaN,NaN,1999.782277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.23895,0.117093,0.233736,0.0,5.2

In [45]:
claim_dme_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital',
                 'claim_is_admit_thru_discharge','claim_is_final','claim_has_external_cause',
                 'claim_admission_year','claim_discharge_year','claim_hospice_start_year',
                 'admission_is_emergency','admission_is_urgent','admission_is_elective','inpatient_ref_from_phys','inpatient_ref_from_clinic',
                  'inpatient_ref_from_transfer','prov_zip','has_employer_subsidy_for_month','claim_duration_in_days',
                 'beneficiary_is_deceased','clm_hha_tot_visit_cnt',
                 'nch_bene_ip_ddctbl_amt','prov_spec_group_is_agencies','prov_spec_group_is_ambulatory_health_care_facilities','prov_spec_group_is_hospital_units',
                  'prov_spec_group_is_hospitals','prov_spec_group_is_nursing_&_custodial_care_facilities','prov_spec_is_adult_mental_health','prov_spec_is_ambulatory_surgical',
                 'prov_spec_is_children','prov_spec_is_community_health','prov_spec_is_critical_access','prov_spec_is_dental','prov_spec_is_emergency_care',
                  'prov_spec_is_esrd','prov_spec_is_federally_qualified_health_center_fqhc','prov_spec_is_health_service','prov_spec_is_multispecialty',
                  'prov_spec_is_mental_health_including_community_mental_health_center','prov_spec_is_oncology','prov_spec_is_physical_therapy',
                 'prov_spec_is_primary_care','prov_spec_is_rehabilitation','prov_spec_is_rural','prov_spec_is_urgent_care']

claim_dme_final = claim_dme.drop(claim_dme_drop,axis=1)
claim_dme_final.columns


Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'clu_anemia', 'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'beneficiary_zip', 'beneficiary_age_at_claim',
       'beneficiary_is_female', 'beneficiary_is_not_white',
       'beneficiary_year_of_coverage_start', 'curr_qualified_old_age_surv',
       'curr_qualified_disability', 'curr_qualified_due_to_renal_disease',
       'enrolled_part_c', 'is_primary_claimant', 'has_rds_cvrg',
       'has_full_dual_status', 'has_partial_dual_status',
       'cost_share_premium_subsidy_percent', 'has_high_copay', 'has_copay',
       'clm_pmt_amt', 'length_of_coverage_at_claim_in_years'],
      dtype='object')

In [46]:
claim_hospice = claim_pred_all_final[claim_pred_all_final['is_hospice_claim']==True]
claim_hospice = claim_hospice.drop(claim_type_filters,axis=1)
claim_hospice.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,1085.0,1085.0,1085.0,1085.0,1085,1085,1085.0,1085.0,1085,1085.0,1085.0,1085.0,1085,1085,1085,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1085.0,1.085000e+03,1085.0,1085,1085.000000,1085,1085,1085.0,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085.0,1085.0,1085.0,1085.0,1085.000000,1085.0,1085.0,1085.0,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,2,<NA>,<NA>,<NA>,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,1,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
top,<NA>,<NA>,<NA>,<NA>,False,False,<NA>,<NA>,True,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,False,False,<NA>,True,False,False,True,True,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,1085,1085,<NA>,<NA>,1052,<NA>,<NA>,<NA>,1085,1085,1085,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1085,NaN,561,679,<NA>,1078,1078,1085,658,830,1082,1082,884,1012,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,1019,1085,1085,1085,1078,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085
mean,-10000930767569.59375,2018.95023,6.617512,2019.024885,NaN,NaN,0.477419,0.522581,NaN,1.0,2019.030415,2018.95023,NaN,NaN,NaN,0.0,0.0,0.0,0.61659,0.56129,0.174194,0.006452,0.508756,0.240553,0.104147,0.009217,0.485714,0.357604,0.326267,0.568664,1.0,0.969585,4.525072e+08,43315.906912,NaN,71.289493,NaN,NaN,2004.451613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.387097,0.130876,0.257143,24.164977,14787.293373,0.0,0.0,14.498618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [47]:
claim_hospice_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital','claim_has_external_cause','claim_admission_year','admission_is_emergency',
                 'admission_is_urgent','admission_is_elective','inpatient_ref_from_phys','inpatient_ref_from_clinic','inpatient_ref_from_transfer',
                  'curr_qualified_old_age_surv','curr_qualified_disability','curr_qualified_due_to_renal_disease',
                  'has_rds_cvrg','has_employer_subsidy_for_month','has_partial_dual_status',
                 'clm_hha_tot_visit_cnt','nch_bene_ip_ddctbl_amt','prov_spec_group_is_ambulatory_health_care_facilities',
                  'prov_spec_group_is_hospital_units','prov_spec_group_is_hospitals',
                  'prov_spec_is_adult_mental_health','prov_spec_is_ambulatory_surgical','prov_spec_is_children','prov_spec_is_community_health',
                  'prov_spec_is_critical_access','prov_spec_is_dental','prov_spec_is_emergency_care','prov_spec_is_federally_qualified_health_center_fqhc',
                 'prov_spec_is_esrd','prov_spec_is_health_service','prov_spec_is_multispecialty',
                  'prov_spec_is_mental_health_including_community_mental_health_center','prov_spec_is_oncology','prov_spec_is_physical_therapy',
                 'prov_spec_is_primary_care','prov_spec_is_rehabilitation','prov_spec_is_rural','prov_spec_is_urgent_care','beneficiary_is_deceased']

claim_hospice_final = claim_hospice.drop(claim_hospice_drop,axis=1)
claim_hospice_final.columns

Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'claim_is_admit_thru_discharge', 'claim_is_final',
       'claim_discharge_year', 'claim_hospice_start_year', 'clu_anemia',
       'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'prov_zip', 'beneficiary_zip',
       'beneficiary_age_at_claim', 'beneficiary_is_female',
       'beneficiary_is_not_white', 'beneficiary_year_of_coverage_start',
       'enrolled_part_c', 'is_primary_claimant', 'has_full_dual_status',
       'cost_share_premium_subsidy_percent', 'has_high_copay', 'has_copay',
       'claim_duration_in_days', 'clm_pmt_amt',
       'length_of_coverage_at_claim_in_years', 'prov_spec_group_is_agencies',
       'prov_spec_grou

In [48]:
claim_hha = claim_pred_all_final[claim_pred_all_final['is_home_health_agency_claim']==True]
claim_hha = claim_hha.drop(claim_type_filters,axis=1)
claim_hha.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,493.0,493.0,493.0,493.0,493,493,493.0,493.0,493,493.0,493.0,493.0,493,493,493,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,493.0,4.930000e+02,493.0,493,493.000000,493,493,493.0,493,493,493,493,493,493,493,493,493,493.0,493.0,493.0,493.0,493.000000,493.0,493.0,493.0,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,2,<NA>,<NA>,<NA>,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,1,2,2,1,1,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
top,<NA>,<NA>,<NA>,<NA>,False,False,<NA>,<NA>,True,<NA>,<NA>,<NA>,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,False,False,<NA>,True,False,False,True,True,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,493,493,<NA>,<NA>,324,<NA>,<NA>,<NA>,493,493,493,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,493,NaN,249,346,<NA>,489,489,493,322,449,493,493,392,469,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,483,493,493,493,492,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493,493
mean,-10000930798671.794922,2018.943205,6.565923,2018.973631,NaN,NaN,0.501014,0.498986,NaN,2018.943205,1.0,1.0,NaN,NaN,NaN,0.0,0.0,0.0,0.588235,0.600406,0.10142,0.006085,0.235294,0.066937,0.087221,0.002028,0.474645,0.312373,0.018256,0.56998,1.0,0.657201,4.925880e+08,44184.206897,NaN,72.134077,NaN,NaN,2005.924949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.058824,0.121704,0.229209,13.555781,8136.211988,15.462475,0.0,13.018256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,415051.061743,2.344519,3.508731,2.355006,NaN,NaN,0.500507,0.500507,NaN

In [49]:
claim_hha_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital','claim_discharge_year','claim_hospice_start_year','admission_is_emergency',
                 'admission_is_urgent','admission_is_elective','inpatient_ref_from_phys','inpatient_ref_from_clinic','inpatient_ref_from_transfer',
                  'curr_qualified_old_age_surv','curr_qualified_disability','curr_qualified_due_to_renal_disease',
                  'has_rds_cvrg','has_employer_subsidy_for_month','has_partial_dual_status',
                 'nch_bene_ip_ddctbl_amt','prov_spec_group_is_ambulatory_health_care_facilities',
                  'prov_spec_group_is_hospital_units','prov_spec_group_is_hospitals',
                  'prov_spec_is_adult_mental_health','prov_spec_is_ambulatory_surgical','prov_spec_is_children','prov_spec_is_community_health',
                  'prov_spec_is_critical_access','prov_spec_is_dental','prov_spec_is_emergency_care','prov_spec_is_federally_qualified_health_center_fqhc',
                 'prov_spec_is_esrd','prov_spec_is_health_service','prov_spec_is_multispecialty',
                  'prov_spec_is_mental_health_including_community_mental_health_center','prov_spec_is_oncology','prov_spec_is_physical_therapy',
                 'prov_spec_is_primary_care','prov_spec_is_rehabilitation','prov_spec_is_rural','prov_spec_is_urgent_care','beneficiary_is_deceased']

claim_hha_final = claim_hha.drop(claim_hha_drop,axis=1)
claim_hha_final.columns

Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'claim_is_admit_thru_discharge', 'claim_is_final',
       'claim_has_external_cause', 'claim_admission_year', 'clu_anemia',
       'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'prov_zip', 'beneficiary_zip',
       'beneficiary_age_at_claim', 'beneficiary_is_female',
       'beneficiary_is_not_white', 'beneficiary_year_of_coverage_start',
       'enrolled_part_c', 'is_primary_claimant', 'has_full_dual_status',
       'cost_share_premium_subsidy_percent', 'has_high_copay', 'has_copay',
       'claim_duration_in_days', 'clm_pmt_amt', 'clm_hha_tot_visit_cnt',
       'length_of_coverage_at_claim_in_years', 'prov_spec_group_is_agencies

In [50]:
claim_snf = claim_pred_all_final[claim_pred_all_final['is_skilled_nursing_facility_claim']==True]
claim_snf = claim_snf.drop(claim_type_filters,axis=1)
claim_snf.describe(include='all')

,clm_id,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_pmt_amt,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care
count,1632.0,1632.0,1632.0,1632.0,1632,1632,1632.0,1632.0,1632,1632.0,1632.0,1632.0,1632,1632,1632,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1632.0,1.632000e+03,1632.0,1632,1632.000000,1632,1632,1632.0,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632.0,1632.0,1632.0,1632.0,1632.000000,1632.0,1632.000000,1632.0,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632
unique,<NA>,<NA>,<NA>,<NA>,1,1,<NA>,<NA>,1,<NA>,<NA>,<NA>,1,1,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1,NaN,2,2,<NA>,2,2,2,2,2,2,2,2,2,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,2,2,2,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,1,1
top,<NA>,<NA>,<NA>,<NA>,False,False,<NA>,<NA>,False,<NA>,<NA>,<NA>,False,False,True,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,False,NaN,True,False,<NA>,True,False,False,True,True,False,False,False,False,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
freq,<NA>,<NA>,<NA>,<NA>,1632,1632,<NA>,<NA>,1632,<NA>,<NA>,<NA>,1632,1632,1632,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,1632,NaN,824,1054,<NA>,1323,1342,1602,994,1455,1625,1626,1316,1528,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN,<NA>,1614,1619,1626,1629,1582,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1632,1629,1632,1632
mean,-10000930760044.865234,2018.879289,6.682598,2018.940564,NaN,NaN,0.518995,0.481005,NaN,1.0,2018.950368,1.0,NaN,NaN,NaN,0.327819,0.321691,0.35049,0.51348,0.496324,0.092525,0.009191,0.261642,0.105392,0.085784,0.008578,0.385417,0.318015,0.026961,0.58701,1.0,0.832721,4.631641e+08,45610.594363,NaN,68.067953,NaN,NaN,2004.943015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.936275,0.127451,0.245711,19.571691,14229.799994,0.0,80.536428,13.936275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [51]:
claim_snf_drop = ['carr_clm_prvdr_asgnmt_bool','claim_is_in_hospital','claim_hospice_start_year','admission_is_emergency',
                 'admission_is_urgent','admission_is_elective',
                  'curr_qualified_due_to_renal_disease',
                  'has_rds_cvrg','has_employer_subsidy_for_month',
                  'prov_spec_is_adult_mental_health','prov_spec_is_ambulatory_surgical','prov_spec_is_children','prov_spec_is_community_health',
                  'prov_spec_is_critical_access','prov_spec_is_dental','prov_spec_is_emergency_care','prov_spec_is_federally_qualified_health_center_fqhc',
                 'prov_spec_is_esrd','prov_spec_is_health_service','prov_spec_is_multispecialty',
                  'prov_spec_is_mental_health_including_community_mental_health_center','prov_spec_is_oncology','prov_spec_is_physical_therapy',
                 'prov_spec_is_primary_care','prov_spec_is_rehabilitation','prov_spec_is_rural','prov_spec_is_urgent_care','beneficiary_is_deceased']

claim_snf_final = claim_snf.drop(claim_snf_drop,axis=1)
claim_snf_final.columns

Index(['clm_id', 'claim_start_year', 'claim_start_month', 'claim_end_year',
       'claim_is_admit_thru_discharge', 'claim_is_final',
       'claim_has_external_cause', 'claim_admission_year',
       'claim_discharge_year', 'inpatient_ref_from_phys',
       'inpatient_ref_from_clinic', 'inpatient_ref_from_transfer',
       'clu_anemia', 'clu_cardiac', 'clu_chronic_kidney', 'clu_depression',
       'clu_diab_complication', 'clu_diabetes',
       'cluster_names_gen_complication', 'clu_encephalopathy',
       'clu_insulin_resistance', 'clu_neonatal_hypertension',
       'clu_severe_kidney', 'clu_stress', 'clu_socio_obese_oth',
       'clu_surg_complication', 'prov_zip', 'beneficiary_zip',
       'beneficiary_age_at_claim', 'beneficiary_is_female',
       'beneficiary_is_not_white', 'beneficiary_year_of_coverage_start',
       'curr_qualified_old_age_surv', 'curr_qualified_disability',
       'enrolled_part_c', 'is_primary_claimant', 'has_full_dual_status',
       'has_partial_dual_status'

# Predictive Modeling
Datasets are finalized for each file type, so it is time to model. For each dataset, I'll divide the set into a 70/30 train/test split, isolate the predictive variable and remove the id, and run a baseline model. Then, I'll tune the parameters for each model and develop a final model. From there, I will attempt to assess key predictive indicators.

For simplicity, each model will use the Random Forest Regressor from SciKit Learn.

In [17]:
scores_dict = {}

In [74]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

claim_ip_y = claim_ip_final[['clm_id','clm_pmt_amt']]
claim_ip_x = claim_ip_final.drop('clm_pmt_amt',axis=1)

ip_x_train, ip_x_test, ip_y_train, ip_y_test = train_test_split(claim_ip_x,claim_ip_y,test_size=0.3,random_state=27)

x_clm_id_tr_ip = ip_x_train['clm_id']
x_clm_id_ts_ip = ip_x_test['clm_id']

ip_x_train = ip_x_train.drop('clm_id',axis=1)
ip_y_train = ip_y_train.drop('clm_id',axis=1)
ip_x_test = ip_x_test.drop('clm_id',axis=1)
ip_y_test = ip_y_test.drop('clm_id',axis=1)

ip_y_test.head()

,clm_pmt_amt
326641,287.82
322853,317.73
225866,165.00
222694,111.01
324503,8744.79


In [ ]:
ip_base_forest = RandomForestRegressor(
    random_state=27)
ip_base_forest.fit(ip_x_train, ip_y_train)
ip_base_preds = ip_base_forest.predict(ip_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
ip_base_score = ip_base_forest.score(ip_x_test,ip_y_test)
ip_base_score

0.5519667719445767

In [ ]:
import numpy as np
from sklearn import metrics

ip_rmse_base = np.sqrt(metrics.mean_squared_error(ip_y_test, ip_base_preds))
ip_rmse_base


8072.966685869264

In [ ]:
scores_dict['inpatient_base'] = {'r_squared': ip_base_score, 'rmse':ip_rmse_base }

I am interested in first understanding the base model's scores for all file types to see if one claim type performs well off the bat.

In [53]:
claim_dme_y = claim_dme_final[['clm_id','clm_pmt_amt']]
claim_dme_x = claim_dme_final.drop('clm_pmt_amt',axis=1)

dme_x_train, dme_x_test, dme_y_train, dme_y_test = train_test_split(claim_dme_x,claim_dme_y,test_size=0.3,random_state=27)

x_clm_id_tr_dme = dme_x_train['clm_id']
y_clm_id_tr_dme = dme_y_train['clm_id']
x_clm_id_ts_dme = dme_x_test['clm_id']
y_clm_id_ts_dme = dme_y_test['clm_id']

dme_x_train = dme_x_train.drop('clm_id',axis=1)
dme_y_train = dme_y_train.drop('clm_id',axis=1)
dme_x_test = dme_x_test.drop('clm_id',axis=1)
dme_y_test = dme_y_test.drop('clm_id',axis=1)

In [ ]:
dme_base_forest = RandomForestRegressor(
    random_state=27)
dme_base_forest.fit(dme_x_train, dme_y_train)
dme_base_preds = dme_base_forest.predict(dme_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
dme_base_score = dme_base_forest.score(dme_x_test,dme_y_test)
dme_base_score

0.0045716886001899715

In [ ]:
dme_rmse_base = np.sqrt(metrics.mean_squared_error(dme_y_test, dme_base_preds))
dme_rmse_base

11.266957574681648

In [ ]:
scores_dict['dme_base'] = {'r_squared': dme_base_score, 'rmse':dme_rmse_base }

In [54]:
claim_hosp_y = claim_hospice_final[['clm_id','clm_pmt_amt']]
claim_hosp_x = claim_hospice_final.drop('clm_pmt_amt',axis=1)

hosp_x_train, hosp_x_test, hosp_y_train, hosp_y_test = train_test_split(claim_hosp_x,claim_hosp_y,test_size=0.3,random_state=27)

x_clm_id_tr_hosp = hosp_x_train['clm_id']
y_clm_id_tr_hosp = hosp_y_train['clm_id']
x_clm_id_ts_hosp = hosp_x_test['clm_id']
y_clm_id_ts_hosp = hosp_y_test['clm_id']

hosp_x_train = hosp_x_train.drop('clm_id',axis=1)
hosp_y_train = hosp_y_train.drop('clm_id',axis=1)
hosp_x_test = hosp_x_test.drop('clm_id',axis=1)
hosp_y_test = hosp_y_test.drop('clm_id',axis=1)

In [ ]:
hosp_base_forest = RandomForestRegressor(
    random_state=27)
hosp_base_forest.fit(hosp_x_train, hosp_y_train)
hosp_base_preds = hosp_base_forest.predict(hosp_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
hosp_base_score = hosp_base_forest.score(hosp_x_test,hosp_y_test)
hosp_base_score

0.9339420111291367

In [ ]:
hosp_rmse_base = np.sqrt(metrics.mean_squared_error(hosp_y_test, hosp_base_preds))
hosp_rmse_base

2932.2825549835156

In [ ]:
scores_dict['hospice_base'] = {'r_squared': hosp_base_score, 'rmse':hosp_rmse_base }

In [55]:
claim_hha_y = claim_hha_final[['clm_id','clm_pmt_amt']]
claim_hha_x = claim_hha_final.drop('clm_pmt_amt',axis=1)

hha_x_train, hha_x_test, hha_y_train, hha_y_test = train_test_split(claim_hha_x,claim_hha_y,test_size=0.3,random_state=27)

x_clm_id_tr_hha = hha_x_train['clm_id']
y_clm_id_tr_hha = hha_y_train['clm_id']
x_clm_id_ts_hha = hha_x_test['clm_id']
y_clm_id_ts_hha = hha_y_test['clm_id']

hha_x_train = hha_x_train.drop('clm_id',axis=1)
hha_y_train = hha_y_train.drop('clm_id',axis=1)
hha_x_test = hha_x_test.drop('clm_id',axis=1)
hha_y_test = hha_y_test.drop('clm_id',axis=1)

In [ ]:
hha_base_forest = RandomForestRegressor(
    random_state=27)
hha_base_forest.fit(hha_x_train, hha_y_train)
hha_base_preds = hha_base_forest.predict(hha_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
hha_base_score = hha_base_forest.score(hha_x_test,hha_y_test)
hha_base_score

0.8013058083023468

In [ ]:
hha_rmse_base = np.sqrt(metrics.mean_squared_error(hha_y_test, hha_base_preds))
hha_rmse_base

1963.3222532005432

In [ ]:
scores_dict['hha_base'] = {'r_squared': hha_base_score, 'rmse':hha_rmse_base}

In [56]:
claim_snf_y = claim_snf_final[['clm_id','clm_pmt_amt']]
claim_snf_x = claim_snf_final.drop('clm_pmt_amt',axis=1)

snf_x_train, snf_x_test, snf_y_train, snf_y_test = train_test_split(claim_snf_x,claim_snf_y,test_size=0.3,random_state=27)

x_clm_id_tr_snf = snf_x_train['clm_id']
y_clm_id_tr_snf = snf_y_train['clm_id']
x_clm_id_ts_snf = snf_x_test['clm_id']
y_clm_id_ts_snf = snf_y_test['clm_id']

snf_x_train = snf_x_train.drop('clm_id',axis=1)
snf_y_train = snf_y_train.drop('clm_id',axis=1)
snf_x_test = snf_x_test.drop('clm_id',axis=1)
snf_y_test = snf_y_test.drop('clm_id',axis=1)

In [ ]:
snf_base_forest = RandomForestRegressor(
    random_state=27)
snf_base_forest.fit(snf_x_train, snf_y_train)
snf_base_preds = snf_base_forest.predict(snf_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
snf_base_score = snf_base_forest.score(snf_x_test,snf_y_test)
snf_base_score

0.7529642567217526

In [ ]:
snf_rmse_base = np.sqrt(metrics.mean_squared_error(snf_y_test, snf_base_preds))
snf_rmse_base

6014.469934056912

In [ ]:
scores_dict['snf_base'] = {'r_squared': snf_base_score, 'rmse':snf_rmse_base}

In [57]:
claim_op_y = claim_op_final[['clm_id','clm_pmt_amt']]
claim_op_x = claim_op_final.drop('clm_pmt_amt',axis=1)

op_x_train, op_x_test, op_y_train, op_y_test = train_test_split(claim_op_x,claim_op_y,test_size=0.3,random_state=27)

x_clm_id_tr_op = op_x_train['clm_id']
y_clm_id_tr_op = op_y_train['clm_id']
x_clm_id_ts_op = op_x_test['clm_id']
y_clm_id_ts_op = op_y_test['clm_id']

op_x_train = op_x_train.drop('clm_id',axis=1)
op_y_train = op_y_train.drop('clm_id',axis=1)
op_x_test = op_x_test.drop('clm_id',axis=1)
op_y_test = op_y_test.drop('clm_id',axis=1)

In [ ]:
op_base_forest = RandomForestRegressor(
    random_state=27)
op_base_forest.fit(op_x_train, op_y_train)
op_base_preds = op_base_forest.predict(op_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
op_base_score = op_base_forest.score(op_x_test,op_y_test)
op_base_score

0.5344227866001088

In [ ]:
op_base_score = op_base_forest.score(op_x_test,op_y_test)
op_base_score
op_rmse_base = np.sqrt(metrics.mean_squared_error(op_y_test, op_base_preds))
op_rmse_base
scores_dict['op_base'] = {'r_squared': op_base_score, 'rmse':op_rmse_base}

4672.203647412223

In [ ]:
scores_dict['op_base'] = {'r_squared': op_base_score, 'rmse':op_rmse_base}

In [ ]:
base_scores = pd.DataFrame(scores_dict).transpose()
base_scores

,r_squared,rmse
inpatient_base,0.551967,8072.966686
dme_base,0.004572,11.266958
hospice_base,0.933942,2932.282555
hha_base,0.801306,1963.322253
snf_base,0.752964,6014.469934
op_base,0.534423,4672.203647


After deploying the base RandomForestRegressor algorithm against the six file types, I observed some mixed results. Some file types, such as hospice, home health, and skilled nursing facilities, performed well with just the base algorithm. Inpatient and Outpatient claims had an R-squared value of 55% and 51%, respectively, which is not desirable performance. The durable medical equipment claims performed the worst, with a negative r-squared value.

I'll perform a grid search to optimize the RandomForestRegressor parameters, hoping to get optimal performance from the hospice, home health, and skilled nursing facility claim types. I also hope to have inpatient and outpatient models to be more explanatory with lower errors. The durable medical equipment claims may not be well-suited to these variables and models, but we'll see what a grid search reveals.

Before tuning the parameters via GridSearch, I want to explore some of the models to see what the parameters are in the base models.

In [ ]:
#looking at select max depth
op_tree_depths = [tree.tree_.max_depth for tree in op_base_forest.estimators_]
max_depth_all_op_trees = max(op_tree_depths)
min_depth_all_op_trees = min(op_tree_depths)

hosp_tree_depths = [tree.tree_.max_depth for tree in hosp_base_forest.estimators_]
max_depth_all_hosp_trees = max(hosp_tree_depths)
min_depth_all_hosp_trees = min(hosp_tree_depths)

dme_tree_depths = [tree.tree_.max_depth for tree in dme_base_forest.estimators_]
max_depth_all_dme_trees = max(dme_tree_depths)
min_depth_all_dme_trees = min(dme_tree_depths)

op_depths = ['outpatient',max_depth_all_op_trees,min_depth_all_op_trees]
hosp_depths = ['hospice',max_depth_all_hosp_trees,min_depth_all_hosp_trees]
dme_depths = ['dme',max_depth_all_dme_trees,min_depth_all_dme_trees]

depth_analysis = pd.DataFrame([op_depths,hosp_depths,dme_depths],columns=['file','max_depth','min_depth'])

depth_analysis

,file,max_depth,min_depth
0,outpatient,65,49
1,hospice,20,14
2,dme,48,35


In [ ]:
from sklearn.model_selection import GridSearchCV

params_dict = {
    'criterion':['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
    'max_depth':[100,75,50,25,10],
    'min_samples_split':[2,10,20,40],
    'min_samples_leaf':[1,5,10,20]
}

In [ ]:
rf = RandomForestRegressor(random_state=27)
grid_search = GridSearchCV(estimator=rf, param_grid=params_dict, cv=5, scoring='r2', verbose=2)

In [ ]:
grid_search.fit(hosp_x_train, hosp_y_train)
best_hosp_params = grid_search.best_params_
best_hosp_model = grid_search.best_estimator_
best_hosp_params

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   0.8s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   0.8s


...

[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.4s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.4s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson',
 'max_depth': 100,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

In [58]:
hosp_best = RandomForestRegressor(criterion='poisson',max_depth=100,min_samples_leaf=1,min_samples_split=2,random_state=27)
tr_hosp_best = hosp_best.fit(hosp_x_train, hosp_y_train)
preds_hosp_test = hosp_best.predict(hosp_x_test)
preds_hosp_tr = hosp_best.predict(hosp_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
hosp_tuned_score = best_hosp_model.score(hosp_x_test, hosp_y_test)
hosp_tuned_score

0.9222973704897915

In [ ]:
hosp_base_preds = best_hosp_model.predict(hosp_x_test)

In [ ]:
hosp_rmse_tuned = np.sqrt(metrics.mean_squared_error(hosp_y_test, hosp_base_preds))
hosp_rmse_tuned

3274.8007316883513

In [ ]:
scores_dict['hosp_tuned'] = {'r_squared': hosp_tuned_score, 'rmse':hosp_rmse_tuned}

In [ ]:
grid_search.fit(hha_x_train, hha_y_train)
best_hha_params = grid_search.best_params_
best_hha_model = grid_search.best_estimator_
best_hha_params

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   0.5s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   0.5s


...

[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.3s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.3s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'absolute_error',
 'max_depth': 100,
 'min_samples_leaf': 5,
 'min_samples_split': 2}

In [63]:
hha_best = RandomForestRegressor(criterion='absolute_error',max_depth=100,min_samples_leaf=5,min_samples_split=2,random_state=27)
tr_hha_best = hha_best.fit(hha_x_train, hha_y_train)
preds_hha_test = hha_best.predict(hha_x_test)
preds_hha_tr = hha_best.predict(hha_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
hha_tuned_score = best_hha_model.score(hha_x_test, hha_y_test)
hha_tuned_score

0.8463389775374596

In [ ]:
hha_base_preds = best_hha_model.predict(hha_x_test)
hha_rmse_tuned = np.sqrt(metrics.mean_squared_error(hha_y_test, hha_base_preds))
hha_rmse_tuned

1726.7180639114993

In [ ]:
scores_dict['hha_tuned'] = {'r_squared': hha_tuned_score, 'rmse':hha_rmse_tuned}

In [ ]:
grid_search.fit(snf_x_train, snf_y_train)
best_snf_params = grid_search.best_params_
best_snf_model = grid_search.best_estimator_
best_snf_params

Fitting 5 folds for each of 320 candidates, totalling 1600 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   1.4s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=squared_error, max_depth=100, min_samples_leaf=1, min_samples_split=2; total time=   1.4s


...

[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.6s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END criterion=poisson, max_depth=10, min_samples_leaf=20, min_samples_split=40; total time=   0.6s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson',
 'max_depth': 100,
 'min_samples_leaf': 5,
 'min_samples_split': 20}

In [66]:
snf_best = RandomForestRegressor(criterion='poisson',max_depth=100,min_samples_leaf=5,min_samples_split=20,random_state=27)
tr_snf_best = snf_best.fit(snf_x_train, snf_y_train)
preds_snf_test = snf_best.predict(snf_x_test)
preds_snf_tr = snf_best.predict(snf_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
snf_tuned_score = best_snf_model.score(snf_x_test, snf_y_test)
snf_tuned_score

0.7449895511533611

In [ ]:
snf_base_preds = best_snf_model.predict(snf_x_test)
snf_rmse_tuned = np.sqrt(metrics.mean_squared_error(snf_y_test, snf_base_preds))
snf_rmse_tuned

6110.77718366444

In [ ]:
scores_dict['snf_tuned'] = {'r_squared': snf_tuned_score, 'rmse':snf_rmse_tuned}

The next three file types - inpatient, outpatient, and DME - have substantively more rows to train than the first three. Therefore, I'm going to limit the number of parameters used in the grid search to limit run time. I'm going to eliminate the max depth parameter entirely, since each of the previous three file types returned a 100 max depth as the best performance. I'll also eliminate the Friedman MSE criterion option and test only the min_samples_split; for the final model, I'll use half of the split value as the min_samples_leaf.

In [ ]:
params_dict_2 = {
    'criterion':['squared_error', 'poisson'],
    'min_samples_split':[2,10,20,40]
}

In [ ]:
rf = RandomForestRegressor(random_state=27)
grid_search = GridSearchCV(estimator=rf, param_grid=params_dict_2, cv=5, scoring='r2', verbose=2)

In [ ]:
grid_search.fit(ip_x_train, ip_y_train)
best_ip_params = grid_search.best_params_
best_ip_model = grid_search.best_estimator_
best_ip_params

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=  15.9s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=  15.8s


...

[CV] END ............criterion=poisson, min_samples_split=40; total time=   9.0s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END ............criterion=poisson, min_samples_split=40; total time=   9.0s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson', 'min_samples_split': 10}

In [75]:
ip_best = RandomForestRegressor(criterion='poisson',min_samples_leaf=5,min_samples_split=10,random_state=27)
tr_ip_best = ip_best.fit(ip_x_train, ip_y_train)
preds_ip_test = ip_best.predict(ip_x_test)
preds_ip_tr = ip_best.predict(ip_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
ip_tuned_score = best_ip_model.score(ip_x_test, ip_y_test)
ip_tuned_score

0.5682646043981089

In [ ]:
ip_base_preds = best_ip_model.predict(ip_x_test)
ip_rmse_tuned = np.sqrt(metrics.mean_squared_error(ip_y_test, ip_base_preds))
ip_rmse_tuned

7924.773819888794

In [ ]:
scores_dict['ip_tuned'] = {'r_squared': ip_tuned_score, 'rmse':ip_rmse_tuned}

In [ ]:
grid_search.fit(dme_x_train, dme_y_train)
best_dme_params = grid_search.best_params_
best_dme_model = grid_search.best_estimator_
best_dme_params

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=  16.1s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=  16.1s


...

[CV] END ............criterion=poisson, min_samples_split=40; total time=  11.3s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END ............criterion=poisson, min_samples_split=40; total time=  11.2s


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson', 'min_samples_split': 40}

In [84]:
dme_best = RandomForestRegressor(criterion='poisson',min_samples_leaf=20,min_samples_split=40,random_state=27)
tr_dme_best = dme_best.fit(dme_x_train, dme_y_train)
preds_dme_test = dme_best.predict(dme_x_test)
preds_dme_tr = dme_best.predict(dme_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
dme_tuned_score = best_dme_model.score(dme_x_test, dme_y_test)
dme_tuned_score

0.09900544428061231

In [ ]:
dme_base_preds = best_dme_model.predict(dme_x_test)
dme_rmse_tuned = np.sqrt(metrics.mean_squared_error(dme_y_test, dme_base_preds))
dme_rmse_tuned

10.719209230927582

In [ ]:
scores_dict['dme_tuned'] = {'r_squared': dme_tuned_score, 'rmse':dme_rmse_tuned}

In [ ]:
grid_search.fit(op_x_train, op_y_train)
best_op_params = grid_search.best_params_
best_op_model = grid_search.best_estimator_
best_op_params

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time= 5.4min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time= 5.4min


...

[CV] END ............criterion=poisson, min_samples_split=40; total time= 4.2min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END ............criterion=poisson, min_samples_split=40; total time= 4.1min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson', 'min_samples_split': 20}

In [87]:
op_best = RandomForestRegressor(criterion='poisson',min_samples_leaf=10,min_samples_split=20,random_state=27)
tr_op_best = op_best.fit(op_x_train, op_y_train)
preds_op_test = op_best.predict(op_x_test)
preds_op_tr = op_best.predict(op_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
op_tuned_score = best_op_model.score(op_x_test, op_y_test)
op_tuned_score

0.5621651758754862

In [ ]:
op_base_preds = best_op_model.predict(op_x_test)
op_rmse_tuned = np.sqrt(metrics.mean_squared_error(op_y_test, op_base_preds))
op_rmse_tuned

4530.864312322797

In [ ]:
scores_dict['op_tuned'] = {'r_squared': op_tuned_score, 'rmse':op_rmse_tuned}

I will also train a general model with all claim types to see if using the individual claim types results in better models.

In [19]:
claim_all_y = claim_pred_all_final[['clm_id','clm_pmt_amt']]
claim_all_x = claim_pred_all_final.drop('clm_pmt_amt',axis=1)

all_x_train, all_x_test, all_y_train, all_y_test = train_test_split(claim_all_x,claim_all_y,test_size=0.3,random_state=27)

x_clm_id_tr = all_x_train['clm_id']
y_clm_id_tr = all_y_train['clm_id']
x_clm_id_ts = all_x_test['clm_id']
y_clm_id_ts = all_y_test['clm_id']

all_x_train = all_x_train.drop('clm_id',axis=1)
all_y_train = all_y_train.drop('clm_id',axis=1)
all_x_test = all_x_test.drop('clm_id',axis=1)
all_y_test = all_y_test.drop('clm_id',axis=1)

all_y_test.head()

,clm_pmt_amt
275114,251.13
324893,165.00
518948,913.45
279163,837.78
193211,1060.88


In [ ]:
all_base_forest = RandomForestRegressor(
    random_state=27)
all_base_forest.fit(all_x_train, all_y_train)
all_base_preds = all_base_forest.predict(all_x_test)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
all_base_score = all_base_forest.score(all_x_test,all_y_test)
all_base_score

0.5299941290781891

In [ ]:
all_rmse_base = np.sqrt(metrics.mean_squared_error(all_y_test, all_base_preds))
all_rmse_base

4697.077781827665

In [ ]:
scores_dict['all_base'] = {'r_squared': all_base_score, 'rmse':all_rmse_base}

In [ ]:
scores_df = pd.DataFrame(scores_dict).transpose()
scores_df

,r_squared,rmse
inpatient_base,0.551967,8072.966686
dme_base,0.004572,11.266958
hospice_base,0.933942,2932.282555
hha_base,0.801306,1963.322253
snf_base,0.752964,6014.469934
op_base,0.534423,4672.203647
hosp_tuned,0.922297,3724.800732
hha_tuned,0.846339,1726.718064
snf_tuned,0.744990,6110.777184
ip_tuned,0.568265,7924.773820


Finally, I'll complete a grid search of all claim types to see if I can improve the scores.

In [22]:
grid_search.fit(all_x_train, all_y_train)
best_all_params = grid_search.best_params_
best_all_model = grid_search.best_estimator_
best_all_params

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=11.2min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END .......criterion=squared_error, min_samples_split=2; total time=10.8min


...

[CV] END ............criterion=poisson, min_samples_split=40; total time= 7.8min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[CV] END ............criterion=poisson, min_samples_split=40; total time= 7.9min


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


{'criterion': 'poisson', 'min_samples_split': 10}

In [122]:
all_best = RandomForestRegressor(criterion='poisson',min_samples_leaf=5,min_samples_split=10,random_state=27)
tr_all_best = all_best.fit(all_x_train, all_y_train)
preds_all_test = all_best.predict(all_x_test)
preds_all_tr = all_best.predict(all_x_train)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [23]:
all_tuned_score = best_all_model.score(all_x_test, all_y_test)
all_tuned_score

0.5806697888553798

In [28]:
import numpy as np
from sklearn import metrics

all_tuned_preds = best_all_model.predict(all_x_test)
all_rmse_tuned = np.sqrt(metrics.mean_squared_error(all_y_test, all_tuned_preds))
all_rmse_tuned

4352.627869241467

In [29]:
scores_dict['all_tuned'] = {'r_squared': all_tuned_score, 'rmse':all_rmse_tuned}

I want to now load the predicted datasets back to BigQuery for storage, use in visualization, and further future analysis.

In [32]:
tuned_x_train = best_all_model.predict(all_x_train)

In [36]:
all_x_train['predicted_y'] = tuned_x_train
all_x_train['actual_y'] = all_y_train

all_x_test['predicted_y'] = all_tuned_preds
all_x_test['actual_y'] = all_y_test

all_claims_final = pd.concat([all_x_train, all_x_test], ignore_index=True)
all_claims_final.head()

,is_inpatient_claim,is_outpatient_claim,is_durable_medical_equipment_claim,is_hospice_claim,is_home_health_agency_claim,is_skilled_nursing_facility_claim,claim_start_year,claim_start_month,claim_end_year,carr_clm_prvdr_asgnmt_bool,claim_is_in_hospital,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,claim_hospice_start_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_is_deceased,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_community_health,prov_spec_is_critical_access,prov_spec_is_dental,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_primary_care,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care,predicted_y,actual_y
0,False,True,False,False,False,False,2016,1,2016,False,True,1,0,True,1,1,1,False,False,False,0,0,0,1,0,1,0,1,1,0,0,1,1,1,0,1,1,631503525,63501,False,46.4,True,False,2014,False,False,True,True,False,False,False,False,False,100,1,1,0,0,0.0,2,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,3134.242315,9018.58
1,False,True,False,False,False,False,2016,9,2016,False,True,1,0,True,1,1,1,False,False,False,0,0,0,1,0,1,0,1,0,0,0,0,1,1,0,1,1,49673707,0,False,61.2,False,False,2013,False,False,True,False,False,False,False,False,False,0,0,0,0,0,0.0,3,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,965.076461,1178.40
2,False,True,False,False,False,False,2022,6,2022,False,True,1,0,True,1,1,1,False,False,False,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,1,26175,26175,False,65.3,True,False,2022,True,False,False,False,True,False,False,False,False,100,0,1,0,0,0.0,0,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,418.261272,475.67
3,False,True,False,False,False,False,2022,1,2022,False,True,1,0,True,1,1,1,False,False,False,0,0,0,1,1,1,0,1,1,0,0,1,1,1,0,1,1,104572545,11369,False,74.1,False,True,2013,True,False,False,True,False,False,False,True,False,100,0,0,0,0,0.0,9,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1248.358208,1533.84
4,False,False,False,False,False,False,2018,1,2018,True,False,0,0,False,1,1,1,False,False,False,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,274056712,27214,False,54.1,True,False,1983,False,True,False,False,True,False,False,False,False,0,0,0,0,0,0.0,35,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,Fal

In [37]:
from pandas_gbq import to_gbq

project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_all_files'

# Write DataFrame to BigQuery
to_gbq(all_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 7096.96it/s]


In [82]:
hosp_x_train['predicted_y'] = preds_hosp_tr
hosp_x_train['actual_y'] = hosp_y_train

hosp_x_test['predicted_y'] = preds_hosp_test
hosp_x_test['actual_y'] = hosp_y_test

hosp_x_train['clm_id'] = x_clm_id_tr_hosp
hosp_x_test['clm_id'] = x_clm_id_ts_hosp

hosp_claims_final = pd.concat([hosp_x_train, hosp_x_test], ignore_index=True)
hosp_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,claim_is_admit_thru_discharge,claim_is_final,claim_discharge_year,claim_hospice_start_year,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,enrolled_part_c,is_primary_claimant,has_full_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_nursing_&_custodial_care_facilities,predicted_y,actual_y,clm_id
0,2015,12,2016,1,0,2016,2015,0,1,0,0,1,0,0,0,0,0,0,0,1,1,752063950,39501,85.1,False,True,1995,False,False,False,0,0,0,30,20,True,False,14848.9250,14041.28,-10000930627186
1,2018,8,2018,0,1,2018,2018,1,0,0,0,1,1,0,0,1,0,0,0,1,1,393013114,39305,75.1,True,False,2008,True,False,False,0,0,0,15,10,True,False,8381.0248,8080.28,-10000930620367
2,2018,8,2018,1,0,2018,2018,1,0,0,0,1,0,0,0,1,1,1,0,1,1,341055028,33914,75.1,False,False,2005,True,False,False,100,0,1,35,13,True,False,23541.4274,24763.20,-10000930151166
3,2020,5,2020,0,1,2020,2020,1,1,0,0,0,0,0,0,1,1,0,0,1,1,750755752,76667,75.1,False,False,2010,False,True,False,0,0,0,15,10,True,False,9936.5878,10421.52,-10000931169702
4,2020,6,2020,0,1,2020,2020,1,1,0,0,1,0,0,0,1,0,1,1,1,1,64603346,83703,65.1,False,False,2020,True,True,False,100,1,1,101,0,True,False,49373.6614,46422.90,-10000930326788


In [83]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_hospice'

# Write DataFrame to BigQuery
to_gbq(hosp_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 8648.05it/s]


In [80]:
hha_x_train['predicted_y'] = preds_hha_tr
hha_x_train['actual_y'] = hha_y_train

hha_x_test['predicted_y'] = preds_hha_test
hha_x_test['actual_y'] = hha_y_test

hha_x_train['clm_id'] = x_clm_id_tr_hha
hha_x_test['clm_id'] = x_clm_id_ts_hha

hha_claims_final = pd.concat([hha_x_train, hha_x_test], ignore_index=True)
hha_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,enrolled_part_c,is_primary_claimant,has_full_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_hha_tot_visit_cnt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_nursing_&_custodial_care_facilities,predicted_y,actual_y,clm_id
0,2019,4,2019,1,0,True,2019,0,0,0,0,0,0,0,0,0,0,0,1,1,1,290711928,21093,80.6,False,True,1959,True,True,False,0,0,0,9,11,60,True,False,6813.78540,7168.61,-10000930523775
1,2015,9,2015,0,1,True,2015,1,1,0,0,0,0,1,0,1,1,0,0,1,1,760136367,0,66.0,False,True,2014,True,True,False,100,0,0,27,29,1,True,False,13690.43195,12688.56,-10000931076829
2,2018,2,2018,1,0,False,2018,0,0,0,0,0,0,0,0,0,0,0,1,1,0,293257556,29611,73.6,True,False,2009,False,True,False,100,0,1,20,22,9,True,False,10576.86595,10284.99,-10000931090032
3,2022,9,2022,1,0,True,2022,1,1,1,0,1,0,0,0,0,0,0,0,1,1,956619120,97103,68.1,True,False,2013,True,True,False,0,0,0,15,17,9,True,False,9144.44740,7766.83,-10000930978377
4,2022,4,2022,0,1,False,2022,0,1,0,0,0,0,0,0,1,1,0,1,1,0,231114517,23124,74.0,True,True,2013,True,True,False,100,0,1,2,3,9,True,False,2586.23090,1473.31,-10000931239538


In [81]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_home_health'

# Write DataFrame to BigQuery
to_gbq(hha_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 7584.64it/s]


In [78]:
snf_x_train['predicted_y'] = preds_snf_tr
snf_x_train['actual_y'] = snf_y_train

snf_x_test['predicted_y'] = preds_snf_test
snf_x_test['actual_y'] = snf_y_test

snf_x_train['clm_id'] = x_clm_id_tr_snf
snf_x_test['clm_id'] = x_clm_id_ts_snf

snf_claims_final = pd.concat([snf_x_train, snf_x_test], ignore_index=True)
snf_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,claim_is_admit_thru_discharge,claim_is_final,claim_has_external_cause,claim_admission_year,claim_discharge_year,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,enrolled_part_c,is_primary_claimant,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,clm_hha_tot_visit_cnt,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,predicted_y,actual_y,clm_id
0,2018,11,2018,0,1,False,1,2018,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,1,0,28403,28328,65.1,True,False,2014,True,False,True,True,False,False,100,1,1,12,0,233.0,4,False,False,False,False,True,9507.275339,8407.14,-10000930845632
1,2019,7,2019,0,1,False,1,2019,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,760631806,76002,105.1,False,False,1979,True,False,True,True,False,False,0,0,0,2,0,0.0,40,False,False,False,False,True,3692.383974,2982.98,-10000931194492
2,2019,11,2019,1,0,False,1,2019,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,117357407,11714,75.1,False,False,2009,True,False,False,True,False,False,0,0,0,27,0,0.0,10,False,False,False,False,True,21894.135120,22759.52,-10000930746255
3,2015,5,2015,1,0,False,1,2015,1,0,0,0,0,0,0,0,0,1,0,1,1,0,1,1,1,199472137,19947,65.1,True,True,2015,True,False,False,True,False,False,0,0,0,7,0,233.0,0,False,False,False,False,True,6727.577654,6723.78,-10000930145509
4,2020,4,2020,0,1,False,1,2020,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,1,1,347863310,0,74.4,True,False,2010,True,False,True,True,False,False,0,0,0,8,0,0.0,10,False,False,False,False,True,7393.525946,7973.26,-10000930197386


In [79]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_skilled_nursing'

# Write DataFrame to BigQuery
to_gbq(snf_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 7695.97it/s]


In [76]:
ip_x_train['predicted_y'] = preds_ip_tr
ip_x_train['actual_y'] = ip_y_train

ip_x_test['predicted_y'] = preds_ip_test
ip_x_test['actual_y'] = ip_y_test

ip_x_train['clm_id'] = x_clm_id_tr_ip
ip_x_test['clm_id'] = x_clm_id_ts_ip

ip_claims_final = pd.concat([ip_x_train, ip_x_test], ignore_index=True)
ip_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,claim_has_external_cause,claim_admission_year,claim_discharge_year,admission_is_emergency,admission_is_urgent,admission_is_elective,inpatient_ref_from_phys,inpatient_ref_from_clinic,inpatient_ref_from_transfer,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,nch_bene_ip_ddctbl_amt,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_ambulatory_surgical,prov_spec_is_critical_access,prov_spec_is_emergency_care,prov_spec_is_federally_qualified_health_center_fqhc,prov_spec_is_health_service,prov_spec_is_primary_care,prov_spec_is_rural,predicted_y,actual_y,clm_id
0,2022,10,2022,False,2022,2022,True,False,False,0,0,1,1,1,1,0,1,0,0,0,1,0,1,1,1,0,201476128,20165,78.0,False,False,2009,True,False,False,True,False,False,False,0,0,0,0,0.0,13,False,False,False,False,True,False,False,False,False,False,False,False,874.596442,1027.72,-10000931240269
1,2023,1,2023,False,2023,2023,True,False,False,0,0,1,1,0,1,0,1,1,1,0,1,1,0,1,1,0,481872687,0,74.7,True,False,2013,True,False,False,False,True,False,False,100,0,0,0,0.0,10,True,False,False,False,False,False,False,False,False,False,False,False,1227.386607,1609.88,-10000930564592
2,2022,5,2022,False,2022,2022,True,False,False,0,0,1,1,0,0,0,1,0,0,0,0,0,0,1,1,0,33401,33704,56.7,False,False,2008,False,True,False,True,True,False,False,0,0,0,0,0.0,14,False,False,False,False,True,False,False,False,False,False,False,False,1438.168911,1662.61,-10000930247606
3,2022,6,2022,True,2022,2022,False,False,True,0,1,0,1,0,0,0,1,1,0,0,1,1,0,0,1,1,917618655,92646,53.1,True,True,2019,False,False,True,True,False,False,False,100,1,1,6,0.0,3,False,False,False,True,False,False,False,False,False,False,False,False,7721.975893,6649.99,-10000931342509
4,2022,8,2022,True,2022,2022,False,False,True,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,900291088,91607,67.4,False,False,2020,True,False,False,True,True,True,False,100,0,0,1,0.0,2,False,True,False,False,False,False,False,False,False,True,False,False,13532.418885,2568.97,-10000931354521


In [77]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_inpatient'

# Write DataFrame to BigQuery
to_gbq(ip_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 7653.84it/s]


In [85]:
dme_x_train['predicted_y'] = preds_dme_tr
dme_x_train['actual_y'] = dme_y_train

dme_x_test['predicted_y'] = preds_dme_test
dme_x_test['actual_y'] = dme_y_test

dme_x_train['clm_id'] = x_clm_id_tr_dme
dme_x_test['clm_id'] = x_clm_id_ts_dme

dme_claims_final = pd.concat([dme_x_train, dme_x_test], ignore_index=True)
dme_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,length_of_coverage_at_claim_in_years,predicted_y,actual_y,clm_id
0,2018,9,2018,0,1,0,0,0,0,1,0,1,1,0,1,1,0,42223,60.3,False,False,1961,False,True,False,True,True,False,True,False,100,0,0,57,7.380593,7.52,-10000930451074
1,2021,5,2021,0,0,0,0,1,0,0,0,0,1,0,1,1,0,98052,60.5,False,True,2017,False,True,False,True,True,False,True,False,100,0,0,4,2.062889,0.00,-10000931287449
2,2020,8,2020,0,0,0,0,1,0,0,0,1,0,0,1,1,0,73135,52.6,False,False,2018,False,True,False,False,True,False,False,False,100,0,1,2,3.332638,0.00,-10000930973397
3,2019,1,2019,0,0,1,0,1,1,0,0,0,0,0,1,1,0,73077,58.5,False,False,1998,False,True,False,True,True,False,False,False,100,1,1,21,3.353296,0.00,-10000930969837
4,2016,4,2016,0,0,0,0,1,0,1,0,1,0,0,1,1,0,0,53.8,True,False,1997,False,True,False,True,True,False,False,False,0,0,0,19,2.106080,0.00,-10000930490645


In [86]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_durable_medical_equipment'

# Write DataFrame to BigQuery
to_gbq(dme_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 9404.27it/s]


In [88]:
op_x_train['predicted_y'] = preds_op_tr
dme_x_train['actual_y'] = op_y_train

op_x_test['predicted_y'] = preds_op_test
op_x_test['actual_y'] = op_y_test

op_x_train['clm_id'] = x_clm_id_tr_op
op_x_test['clm_id'] = x_clm_id_ts_op

op_claims_final = pd.concat([op_x_train, op_x_test], ignore_index=True)
op_claims_final.head()

,claim_start_year,claim_start_month,claim_end_year,claim_has_external_cause,clu_anemia,clu_cardiac,clu_chronic_kidney,clu_depression,clu_diab_complication,clu_diabetes,cluster_names_gen_complication,clu_encephalopathy,clu_insulin_resistance,clu_neonatal_hypertension,clu_severe_kidney,clu_stress,clu_socio_obese_oth,clu_surg_complication,prov_zip,beneficiary_zip,beneficiary_age_at_claim,beneficiary_is_female,beneficiary_is_not_white,beneficiary_year_of_coverage_start,curr_qualified_old_age_surv,curr_qualified_disability,curr_qualified_due_to_renal_disease,enrolled_part_c,is_primary_claimant,has_rds_cvrg,has_employer_subsidy_for_month,has_full_dual_status,has_partial_dual_status,cost_share_premium_subsidy_percent,has_high_copay,has_copay,claim_duration_in_days,length_of_coverage_at_claim_in_years,prov_spec_group_is_agencies,prov_spec_group_is_ambulatory_health_care_facilities,prov_spec_group_is_hospital_units,prov_spec_group_is_hospitals,prov_spec_group_is_nursing_&_custodial_care_facilities,prov_spec_is_adult_mental_health,prov_spec_is_ambulatory_surgical,prov_spec_is_children,prov_spec_is_critical_access,prov_spec_is_emergency_care,prov_spec_is_esrd,prov_spec_is_mental_health_including_community_mental_health_center,prov_spec_is_multispecialty,prov_spec_is_oncology,prov_spec_is_physical_therapy,prov_spec_is_rehabilitation,prov_spec_is_rural,prov_spec_is_urgent_care,predicted_y,clm_id,actual_y
0,2016,12,2016,True,0,1,0,0,0,0,0,0,0,0,0,0,1,1,115901740,11003,78.4,True,False,1987,True,False,False,False,True,False,False,False,False,0,0,0,0,29,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1576.773248,-10000930765292,NaN
1,2018,1,2018,True,0,1,1,0,0,0,0,0,1,0,1,1,1,1,630173417,63376,86.2,False,False,1996,True,False,False,False,False,False,False,False,False,100,1,1,0,22,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1023.929774,-10000930656624,NaN
2,2016,8,2016,True,0,0,0,0,0,0,0,0,0,0,0,1,1,1,982231642,98273,26.3,True,False,2001,False,True,False,False,True,False,False,True,False,100,0,0,0,15,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,18021.082176,-10000931297436,NaN
3,2016,7,2016,True,1,0,1,0,1,1,0,0,1,0,0,1,1,1,191251012,19125,55.6,True,False,1994,False,True,False,True,True,False,False,False,False,100,0,0,0,22,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1028.870339,-10000931024928,NaN
4,2019,2,2019,True,1,1,1,1,1,1,1,1,1,1,1,1,1,1,60463,60453,67.2,False,True,2016,True,False,False,True,True,False,False,False,False,0,0,0,0,3,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,1029.736197,-10000930338231,NaN


In [89]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_predictors_outpatient'

# Write DataFrame to BigQuery
to_gbq(op_claims_final, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 7489.83it/s]


The final step is to assess feature importances of the final model states.

In [99]:
hosp_feat = hosp_best.feature_importances_
hosp_feat

array([7.42734381e-04, 2.70496773e-03, 8.76057882e-04, 3.59715005e-04,
       4.37061340e-04, 9.50030544e-04, 6.65778450e-04, 5.97359647e-04,
       4.17101511e-04, 3.80757826e-04, 1.27298509e-05, 3.68821272e-04,
       5.58482565e-04, 1.98553309e-04, 1.56656084e-05, 6.66447411e-04,
       5.38886185e-04, 4.82616697e-04, 4.50758594e-04, 0.00000000e+00,
       5.02845114e-05, 8.09856864e-03, 9.71297733e-03, 5.90944522e-04,
       4.78210127e-04, 1.96333431e-03, 3.24585814e-03, 4.81568900e-04,
       3.59654066e-04, 4.57118572e-04, 3.97211628e-04, 2.36622756e-04,
       4.35047319e-04, 9.60039244e-01, 1.79782833e-03, 2.28499412e-04,
       2.50153958e-06])

In [100]:
hosp_data = []
for index,column in enumerate(hosp_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['hospice',column,hosp_feat[index]]
    hosp_data.append(row)

hosp_fi_df = pd.DataFrame(data=hosp_data,columns=['file_type','column_name','feature_importance'])
hosp_fi_df.head()

,file_type,column_name,feature_importance
0,hospice,claim_start_year,0.000743
1,hospice,claim_start_month,0.002705
2,hospice,claim_end_year,0.000876
3,hospice,claim_is_admit_thru_discharge,0.000360
4,hospice,claim_is_final,0.000437


In [102]:
hosp_fi_df[hosp_fi_df['feature_importance']==max(hosp_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
33,hospice,claim_duration_in_days,0.960039


In [103]:
hha_feat = hha_best.feature_importances_

In [107]:
hha_data = []
for index,column in enumerate(hha_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['hha',column,hha_feat[index]]
    hha_data.append(row)

hha_fi_df = pd.DataFrame(data=hha_data,columns=['file_type','column_name','feature_importance'])
hha_fi_df.head()

,file_type,column_name,feature_importance
0,hha,claim_start_year,0.001148
1,hha,claim_start_month,0.015460
2,hha,claim_end_year,0.002480
3,hha,claim_is_admit_thru_discharge,0.002283
4,hha,claim_is_final,0.001979


In [108]:
hha_fi_df[hha_fi_df['feature_importance']==max(hha_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
34,hha,clm_hha_tot_visit_cnt,0.600414


In [109]:
snf_feat = snf_best.feature_importances_

In [110]:
snf_data = []
for index,column in enumerate(snf_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['snf',column,snf_feat[index]]
    snf_data.append(row)

snf_fi_df = pd.DataFrame(data=snf_data,columns=['file_type','column_name','feature_importance'])
snf_fi_df.head()

,file_type,column_name,feature_importance
0,snf,claim_start_year,0.001405
1,snf,claim_start_month,0.003425
2,snf,claim_end_year,0.001303
3,snf,claim_is_admit_thru_discharge,0.000174
4,snf,claim_is_final,0.000141


In [111]:
snf_fi_df[snf_fi_df['feature_importance']==max(snf_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
40,snf,claim_duration_in_days,0.937776


In [113]:
ip_feat = ip_best.feature_importances_

In [114]:
ip_data = []
for index,column in enumerate(ip_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['inpatient',column,ip_feat[index]]
    ip_data.append(row)

ip_fi_df = pd.DataFrame(data=ip_data,columns=['file_type','column_name','feature_importance'])
ip_fi_df.head()

,file_type,column_name,feature_importance
0,inpatient,claim_start_year,0.005429
1,inpatient,claim_start_month,0.026532
2,inpatient,claim_end_year,0.005014
3,inpatient,claim_has_external_cause,0.002814
4,inpatient,claim_admission_year,0.005961


In [115]:
ip_fi_df[ip_fi_df['feature_importance']==max(ip_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
6,inpatient,admission_is_emergency,0.291826


In [116]:
op_feat = op_best.feature_importances_

In [117]:
op_data = []
for index,column in enumerate(op_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['outpatient',column,op_feat[index]]
    op_data.append(row)

op_fi_df = pd.DataFrame(data=op_data,columns=['file_type','column_name','feature_importance'])
op_fi_df.head()

,file_type,column_name,feature_importance
0,outpatient,claim_start_year,0.014721
1,outpatient,claim_start_month,0.033179
2,outpatient,claim_end_year,0.016489
3,outpatient,claim_has_external_cause,0.000570
4,outpatient,clu_anemia,0.006271


In [118]:
op_fi_df[op_fi_df['feature_importance']==max(op_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
20,outpatient,beneficiary_age_at_claim,0.318021


In [119]:
dme_feat = dme_best.feature_importances_

In [120]:
dme_data = []
for index,column in enumerate(dme_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['dme',column,dme_feat[index]]
    dme_data.append(row)

dme_fi_df = pd.DataFrame(data=dme_data,columns=['file_type','column_name','feature_importance'])
dme_fi_df.head()

,file_type,column_name,feature_importance
0,dme,claim_start_year,0.019015
1,dme,claim_start_month,0.056797
2,dme,claim_end_year,0.018925
3,dme,clu_anemia,0.020548
4,dme,clu_cardiac,0.008189


In [121]:
dme_fi_df[dme_fi_df['feature_importance']==max(dme_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
18,dme,beneficiary_age_at_claim,0.275633


In [123]:
all_feat = all_best.feature_importances_

In [124]:
all_data = []
for index,column in enumerate(all_claims_final.columns):
  if column not in ['predicted_y','actual_y','clm_id']:
    row = ['all_files',column,all_feat[index]]
    all_data.append(row)

all_fi_df = pd.DataFrame(data=all_data,columns=['file_type','column_name','feature_importance'])
all_fi_df.head()

,file_type,column_name,feature_importance
0,all_files,is_inpatient_claim,2.082916e-11
1,all_files,is_outpatient_claim,3.268286e-11
2,all_files,is_durable_medical_equipment_claim,1.058014e-12
3,all_files,is_hospice_claim,6.602965e-13
4,all_files,is_home_health_agency_claim,0.000000e+00


In [125]:
all_fi_df[all_fi_df['feature_importance']==max(all_fi_df['feature_importance'])]

,file_type,column_name,feature_importance
61,all_files,prov_spec_group_is_ambulatory_health_care_faci...,1.007725e-07


In [126]:
fi_df_comprehensive = pd.concat([all_fi_df, ip_fi_df, op_fi_df, hosp_fi_df, hha_fi_df, snf_fi_df, dme_fi_df], ignore_index=True)

In [127]:
project_id = 'spatial-earth-449020-m3'
table_name = 'lexie_custom_datasets.final_feat_imps_all_files'

# Write DataFrame to BigQuery
to_gbq(fi_df_comprehensive, table_name, project_id=project_id, if_exists='replace')

100%|██████████| 1/1 [00:00<00:00, 5584.96it/s]
